# MLIP Active-Learning Tutorial

This tutorial shows how the ALF `MLIPModel` (a MACE machine-learned interatomic
potential) is used in an **online active-learning loop**. Our use case: starting from a
pretrained foundation model and **finetuning** it into an accurate force field for a single
organic molecule, while spending as few expensive labels as possible.

### How this differs from the design tutorials

The protein-design tutorials *maximize a fitness*. Here we do the opposite kind of active
learning: we **minimize model error using as few expensive labels as possible**. Each label is a
quantum-chemistry calculation (DFT in production; here a fast GFN2-xTB stand-in). The model picks
which conformers to label by where its committee of models *disagrees most* — the most
informative structures.

### Experiment overview

1. Download a pretrained MACE organics model and finetune it on a few conformers of a molecule.
2. Use a committee (ensemble) of finetuned models to estimate prediction uncertainty.
3. Each round: propose perturbed conformers, pick the most uncertain ones, label them with the
   xTB oracle, and finetune again.
4. Compare uncertainty-driven acquisition against a random baseline on a learning curve.

### Framework Components

1. **Dataset** (`ConformerDataset`, defined below): holds the molecule's conformers and their
   energies, and splits them into train/validation/test. The candidate pool is unused — new
   conformers are generated on the fly.
2. **Surrogate Model** ([`MLIPModel`](https://instadeepai.github.io/alf/api/alf_tools/models/)) wrapped in an [`EnsembleWrapper`](https://instadeepai.github.io/alf/api/alf_tools/models/) committee: each member finetunes the pretrained MACE model; their disagreement is our uncertainty.
3. **Search Strategy** ([`ProtocolSearch`](https://instadeepai.github.io/alf/api/alf_core/optimizer/search/) + `RattleSearch`): generates new conformers by perturbing current training structures.
4. **Acquisition Function** ([`UCB`](https://instadeepai.github.io/alf/api/alf_tools/optimizer/acquisition_functions/)): selects conformers with the highest committee disagreement.
5. **Optimizer** ([`Optimizer`](https://instadeepai.github.io/alf/api/alf_core/optimizer/optimizer/)): handles the ask/tell cycle.
6. **Oracle** ([`Oracle`](https://instadeepai.github.io/alf/api/alf_core/oracle/) + `XTBScorer`): computes ground-truth energy and forces with GFN2-xTB.
7. **Task** ([`DesignTask`](https://instadeepai.github.io/alf/api/alf_core/tasks/design_task/)): orchestrates the active-learning loop.

### Step 0: Environment Setup

Create the environment with `uv sync` from the `tutorials/` directory (CPU PyTorch by default).
This tutorial additionally needs `tblite` (the GFN2-xTB engine used as our oracle) and downloads a
pretrained MACE model from Hugging Face (public, no credentials).

In [ ]:
import subprocess
import sys
from pathlib import Path

from huggingface_hub import hf_hub_download

# Install the xTB engine used as the oracle (real, fast, deterministic QM) and s3fs,
# which is needed because alf_tools.models.utils.mlip_utils initializes an fsspec S3
# filesystem at import time.
subprocess.check_call(["uv", "pip", "install", "--python", sys.executable, "tblite", "s3fs"])

# Download the pretrained MACE organics model into the alf model directory so the
# MLIPModel loader finds it locally and skips its (private) S3 fallback.
import alf_tools.models.utils.mlip_utils as mlip_utils

models_dir = mlip_utils._MODELS_DIR
models_dir.mkdir(parents=True, exist_ok=True)
hf_hub_download(
    repo_id="InstaDeepAI/mlip_models_organics_v2",
    filename="mace_organics_02.zip",
    local_dir=str(models_dir),
)
print(f"✅ Pretrained model present at {models_dir / 'mace_organics_02.zip'}")

### Step 1: Import Required Libraries

In [ ]:
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # macOS: tblite and torch both load OpenMP

import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ase import Atoms
from ase.build import molecule
from tblite.ase import TBLite

from alf_core import (
    AcquisitionFunction,
    BaseModel,
    Candidate,
    DesignTask,
    FileStateLogger,
    LabelledCandidates,
    Optimizer,
    Oracle,
    Predictions,
    ProtocolSearch,
    SearchProtocol,
    State,
    Surrogate,
    TerminalStateLogger,
)
from alf_core.dataclasses.candidate import Modality
from alf_core.dataset.base_dataset import BaseDataset, BaseDatasetConfig
from alf_core.utils.enums import ProblemType
from alf_tools.models.ensemble import EnsembleWrapper, EnsembleWrapperConfig, SubsampleConfig
from alf_tools.models.mlip import MLIPModel, MLIPModelConfig, MLIPTrainConfig
from alf_tools.optimizer.acquisition_functions import UCB

print("✅ All imports successful!")

### Step 2: Build the Molecule and Initial Dataset

We take a single ethanol molecule, generate ~30 rattled conformers, and label each with GFN2-xTB
(energy and forces). These become an ALF dataset split into train/validation/test. The energy is
the regression target; forces are stored in each candidate's `features` so the MACE model can
train on them.

In [ ]:
def xtb_energy_forces(atoms: Atoms) -> tuple[float, np.ndarray]:
    """Return GFN2-xTB potential energy (eV) and forces (N, 3) for `atoms`."""
    atoms = atoms.copy()
    atoms.calc = TBLite(method="GFN2-xTB", verbosity=0)
    return float(atoms.get_potential_energy()), np.asarray(atoms.get_forces())


def make_conformers(base: Atoms, n: int, stdevs, seed: int) -> list[Atoms]:
    """Return `n` rattled copies of `base`, cycling through `stdevs` for variety."""
    confs = []
    for i in range(n):
        a = base.copy()
        a.rattle(stdev=stdevs[i % len(stdevs)], seed=seed + i)
        confs.append(a)
    return confs


def build_labelled(confs: list[Atoms]) -> LabelledCandidates:
    """Label conformers with xTB; energy is the label, forces go in features."""
    candidates, labels = [], []
    for a in confs:
        energy, forces = xtb_energy_forces(a)
        candidates.append(Candidate(data=a, modality=Modality.STRUCTURE, features={"forces": forces}))
        labels.append(energy)
    return LabelledCandidates(candidates=candidates, labels=np.asarray(labels))


class ConformerDataset(BaseDataset):
    """In-notebook dataset of pre-labelled molecular conformers."""

    def __init__(self, config: BaseDatasetConfig, labelled: LabelledCandidates):
        """Store the pre-labelled conformers and initialise the base dataset."""
        super().__init__(config)
        self._labelled = labelled

    def load_dataset(self) -> LabelledCandidates:
        """Return the pre-labelled conformers as the raw dataset."""
        return self._labelled


base_molecule = molecule("CH3CH2OH")
conformers = make_conformers(base_molecule, n=30, stdevs=[0.03, 0.06, 0.10], seed=0)
labelled_conformers = build_labelled(conformers)

dataset = ConformerDataset(
    BaseDatasetConfig(
        name="ethanol_conformers",
        modality=Modality.STRUCTURE,
        seed=51505,
        train_ratio=0.4,
        validation_frac=0.2,
        test_ratio=0.3,
        split_type="random",
        problem_type=ProblemType.REGRESSION,
    ),
    labelled_conformers,
)
dataset.setup()

print("✅ Dataset initialized!")
print(f"   - Molecule: ethanol ({len(base_molecule)} atoms)")
print(f"   - Train / val / test: {len(dataset.train_dataset)} / "
      f"{len(dataset.validation_dataset)} / {len(dataset.test_dataset)}")
print(f"   - Energy range (eV): {labelled_conformers.labels.min():.3f} to "
      f"{labelled_conformers.labels.max():.3f}")

### Step 3: The Oracle (GFN2-xTB)

The oracle is the expensive ground-truth evaluator. In production this is DFT or a wet-lab
measurement; here we use GFN2-xTB — a fast, deterministic semi-empirical quantum method that
returns energy and forces. `XTBScorer` wraps it as an ALF `BaseModel`; when the `Oracle` evaluates
acquired conformers, it records both the energy (the label) and the forces (in `features`).

In [ ]:
class XTBScorer(BaseModel):
    """Oracle that scores conformers with GFN2-xTB, the 'expensive' ground truth."""

    def predict(self, candidate_points: list[Candidate]) -> Predictions:
        """Return xTB energies as means and store xTB forces on each candidate."""
        energies = []
        for cand in candidate_points:
            energy, forces = xtb_energy_forces(cand.data)
            if cand.features is None:
                cand.features = {}
            cand.features["forces"] = forces
            energies.append(energy)
        return Predictions(means=np.asarray(energies))

    def featurise(self, inputs):
        """No-op: the oracle does not featurise."""
        return inputs

    def train(self, train_data, val_data, metrics_collector=None):
        """No-op: the oracle is not trained."""
        return None

    def sample(self, condition=None):
        """Not implemented for the oracle."""
        raise NotImplementedError("XTBScorer does not sample.")


oracle = Oracle(scorer=XTBScorer())
print("✅ Oracle initialized (GFN2-xTB)!")